# RAG Chatbot Development and Evaluation

**ITC-C508 (Deep Learning and NLP) — Final Project** · **Group Purple R**
Mark Emmanuel P. Sagarino · Ram Kelly Jerzen Leliza · Sean John C. Dizon
College of Computer Studies, Jose Rizal University

---

### Domain

Philippine consumer-remedy law for defective purchases: four statutes and the Department
of Trade and Industry's published guidance, 73,652 characters across five documents.

A buyer with a defective purchase is governed by one of four different statutes, and they
do not give the same answer. Deciding *which* regime applies is the hard part, and it is
what makes this a worthwhile retrieval problem rather than a lookup.

### What this notebook contains

| Section | What it does | Deliverable |
|---|---|---|
| 1–3 | Dependencies, configuration, data upload | — |
| 4 | Pre-flight check on the test set, before anything expensive runs | — |
| 5–8 | Document processing, chunking, embeddings, vector store, LLM, RAG function | 3 |
| 9 | Device verification — where the weights actually sit | — |
| 10 | A single end-to-end call to prove the pipeline works | 3 |
| 11 | The twelve-question test set and the scoring functions | — |
| 12–13 | Chunking experiment across five configurations, and the selection | 2 |
| 14 | Full evaluation of all twelve questions | 5 |
| 15 | Experiment logs and checkpoints | 4 |
| 16 | The working chatbot at the selected configuration | 3 |
| 17 | Downloads | — |

### How scoring works, stated once

Two rules govern every number in this notebook. Both are decisions, not defaults, and
both are implemented in Section 10.

**An evidence anchor is a verbatim string, not a chunk number.** A retrieved chunk counts
as relevant when its text *contains* one of the anchors for that question. Chunk numbers
change whenever the chunk size changes, and Section 12 varies the chunk size on purpose,
so chunk numbers cannot serve as ground truth. Anchors can.

**Undefined is not zero.** An unanswerable question has no anchors and no answer keys, so
recall, precision and answer quality are *undefined* for it — not zero. Writing zero
would drag the mean down by a quarter for no reason. Those three columns stay blank for
U1–U3, and refusal behaviour is reported separately in its own column.

### Authorship of components

The evaluation test set in `PurpleR_TestSet.csv`, its reference answers and its 21
evidence anchors were authored by R. K. J. Leliza. The pre-flight check in Section 7 was
authored by S. J. C. Dizon, who also owns the execution logs in Section 15. The pipeline,
the experiment and the evaluation harness were written by M. E. P. Sagarino.

## 1. Install dependencies

Pinned below the LangChain 1.0 split, which moved several import paths. `-q` keeps the
output short; the versions actually resolved are recorded in Section 15 so the run can be
reproduced.

In [1]:
%pip -q install -U "langchain<1.0" "langchain-community<0.4" langchain-chroma \
    langchain-huggingface sentence-transformers chromadb transformers accelerate \
    openpyxl 2>/dev/null
print("dependencies installed")

dependencies installed


## 2. Configure

Every tunable in one place. Sections 12 and 13 override the chunking values while they
sweep; everything else holds for the whole run.

`DO_SAMPLE = False` is not a performance setting. Greedy decoding makes the run
repeatable, which matters because Section 15 is an audit record — a sampled answer could
not be checked against the report afterwards.

In [2]:
from pathlib import Path
import time

# ---------- identity ----------
GROUP_NAME = "PurpleR"
DOMAIN_NAME = "Philippine consumer-remedy law for defective purchases"

# ---------- data ----------
DATA_DIR = "/content/data"
SOURCE_DIR = f"{DATA_DIR}/knowledge_base"
TESTSET_CSV = f"{DATA_DIR}/PurpleR_TestSet.csv"

# ---------- indexing (starting point; section 12 sweeps these) ----------
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "purpler_final"

# ---------- retrieval ----------
TOP_K = 3

# ---------- generation ----------
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 200
DO_SAMPLE = False          # deterministic: an audit record must be reproducible
DEVICE_MAP = "auto"

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(SOURCE_DIR).mkdir(parents=True, exist_ok=True)

print("Group          :", GROUP_NAME)
print("Domain         :", DOMAIN_NAME)
print("Embedding model:", EMBEDDING_MODEL)
print("Language model :", LLM_MODEL)
print("Chunk / overlap:", CHUNK_SIZE, "/", CHUNK_OVERLAP, "| top-k:", TOP_K)
print("Max new tokens :", MAX_NEW_TOKENS, "| sampling:", DO_SAMPLE)

Group          : PurpleR
Domain         : Philippine consumer-remedy law for defective purchases
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Language model : Qwen/Qwen2.5-3B-Instruct
Chunk / overlap: 800 / 100 | top-k: 3
Max new tokens : 200 | sampling: False


## 3. Upload the data package

Upload **`PurpleR_FinalProject_Data.zip`**.

Not `PurpleR_RAG_Experiment_Data.zip` — that is the earlier WW-F1/F2 archive, it carries
ten questions instead of twelve, and the acceptance check in Section 7 will reject it.
Windows hides the extension, so look for the folder icon with a zipper.

The cell is idempotent: if the corpus and the test set are already in place it skips the
upload, so *Run all* after a partial run does not stop to ask again.

In [3]:
import zipfile, shutil, os, glob
from google.colab import files


def have_data():
    return len(glob.glob(f"{SOURCE_DIR}/*.txt")) > 0 and os.path.exists(TESTSET_CSV)


if have_data():
    print("Corpus and test set already present; skipping upload.")
else:
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                z.extractall("/content/_unzip")
            for src in glob.glob("/content/_unzip/**/*", recursive=True):
                if not os.path.isfile(src):
                    continue
                base = os.path.basename(src)
                if base.endswith(".txt") and "knowledge_base" in src:
                    shutil.copy(src, f"{SOURCE_DIR}/{base}")
                elif base in ("PurpleR_TestSet.csv", "purpler_testset_check.py",
                              "verify_anchors.py", "TestSet_Rationale.md", "README.txt"):
                    shutil.copy(src, f"{DATA_DIR}/{base}")

print("\nKnowledge base:")
total = 0
for p in sorted(glob.glob(f"{SOURCE_DIR}/*.txt")):
    size = os.path.getsize(p)
    total += size
    print(f"  {os.path.basename(p):<46}{size:>8,} bytes")
print(f"  {'TOTAL':<46}{total:>8,} bytes")

if not have_data():
    raise RuntimeError("Upload PurpleR_FinalProject_Data.zip and run this cell again.")
print("\nPurpleR_TestSet.csv present.")

Saving PurpleR_FinalProject_Data.zip to PurpleR_FinalProject_Data.zip

Knowledge base:
  01_RA7394_Consumer_Act.txt                      18,150 bytes
  02_RA8792_ECommerce_Act.txt                     13,248 bytes
  03_RA10642_Lemon_Law.txt                        15,785 bytes
  04_RA11967_Internet_Transactions_Act.txt        18,878 bytes
  05_DTI_Consumer_Guidance.txt                     7,591 bytes
  TOTAL                                           73,652 bytes

PurpleR_TestSet.csv present.


## 4. Pre-flight check on the test set

**Authored by S. J. C. Dizon.**

Forty-seven mechanical checks over the test set, run before anything expensive starts: the
corpus size, the row counts, that every anchor occurs exactly once in the corpus and sits
inside a single line, that the unanswerable rows carry nothing to score against, that
every reference answer scores 1.00 against its own keys, and that a content-free answer
which only name-drops the statutes scores 0.00.

Two of those are worth explaining, because they are the ones that have actually caught
things.

*Every gold answer must score 1.00.* If the reference answer cannot satisfy its own keys,
no chatbot can either, and the question is unwinnable. This caught `excluded` as an
answer-key variant where the reference answer says "excludes" — the stem `exclud` matches
both.

*A content-free answer must score 0.00.* An answer that names the right statutes and says
nothing else was scoring 1.00 on an easy question, because the bare variant `4` matched
inside "7394". Answer keys that are purely digits are now matched on word boundaries.

The cell aborts the run on failure. A broken test set discovered forty minutes into a run
costs the whole run.

In [4]:
import subprocess, sys, shutil

check_script = f"{DATA_DIR}/purpler_testset_check.py"
if os.path.exists(check_script):
    # the script expects the test set and knowledge_base/ side by side
    result = subprocess.run([sys.executable, check_script, DATA_DIR],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    PREFLIGHT_PASSED = (result.returncode == 0)
    PREFLIGHT_OUTPUT = result.stdout
    if not PREFLIGHT_PASSED:
        raise RuntimeError("Pre-flight check failed. Fix the test set before running.")
else:
    PREFLIGHT_PASSED = None
    PREFLIGHT_OUTPUT = "purpler_testset_check.py not found in the data package."
    print(PREFLIGHT_OUTPUT)

print("Pre-flight:", "passed" if PREFLIGHT_PASSED else PREFLIGHT_PASSED)


[1] corpus and row counts
  ok   corpus is 73,652 characters  -- got 73,652
  ok   12 rows  -- got 12
  ok   9 answerable  -- got 9
  ok   3 unanswerable  -- got 3

[2,3] evidence anchors - unique in corpus, and inside one line
  ok   E1 anchor 1  -- occurrences=1 single_line=True len=33
  ok   E2 anchor 1  -- occurrences=1 single_line=True len=30
  ok   E3 anchor 1  -- occurrences=1 single_line=True len=48
  ok   E3 anchor 2  -- occurrences=1 single_line=True len=29
  ok   A1 anchor 1  -- occurrences=1 single_line=True len=60
  ok   A1 anchor 2  -- occurrences=1 single_line=True len=54
  ok   A1 anchor 3  -- occurrences=1 single_line=True len=41
  ok   A2 anchor 1  -- occurrences=1 single_line=True len=62
  ok   A2 anchor 2  -- occurrences=1 single_line=True len=77
  ok   A3 anchor 1  -- occurrences=1 single_line=True len=41
  ok   A3 anchor 2  -- occurrences=1 single_line=True len=43
  ok   A3 anchor 3  -- occurrences=1 single_line=True len=45
  ok   D1 anchor 1  -- occurrences=1 si

## 5. Load the documents and split them into chunks

Splitting is wrapped in a function because Section 12 calls it repeatedly at different
chunk sizes. The documents are loaded from disk once and reused, so a difference between
configurations cannot be caused by re-reading the files.

`RecursiveCharacterTextSplitter` tries paragraph breaks first, then sentences, then words,
and only cuts mid-word as a last resort. `add_start_index=True` records where each chunk
began in its source document, which is what lets a retrieved chunk be traced back to a
location in the corpus rather than just a number.

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []
for path in sorted(glob.glob(f"{SOURCE_DIR}/*.txt")):
    documents.extend(TextLoader(path, encoding="utf-8").load())

SOURCE_CHARS = sum(len(d.page_content) for d in documents)
print(f"Loaded {len(documents)} documents, {SOURCE_CHARS:,} characters\n")

DOC_STATS = []
for d in documents:
    name = os.path.basename(d.metadata.get("source", "?"))
    DOC_STATS.append({"Document": name, "Characters": len(d.page_content)})
    print(f"  {name:<46}{len(d.page_content):>8,} chars")


def split_documents(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, add_start_index=True)
    t0 = time.perf_counter()
    chunks = splitter.split_documents(docs)
    return chunks, time.perf_counter() - t0


chunks, split_seconds = split_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
mean_chars = sum(len(c.page_content) for c in chunks) / len(chunks)
print(f"\nAt {CHUNK_SIZE}/{CHUNK_OVERLAP}: {len(chunks)} chunks, "
      f"mean {mean_chars:.0f} chars, split in {split_seconds:.3f} s")

Loaded 5 documents, 73,652 characters

  01_RA7394_Consumer_Act.txt                      18,150 chars
  02_RA8792_ECommerce_Act.txt                     13,248 chars
  03_RA10642_Lemon_Law.txt                        15,785 chars
  04_RA11967_Internet_Transactions_Act.txt        18,878 chars
  05_DTI_Consumer_Guidance.txt                     7,591 chars

At 800/100: 125 chunks, mean 596 chars, split in 0.003 s


## 6. Embeddings and the vector store

`all-MiniLM-L6-v2` maps each chunk to a 384-dimensional vector; Chroma stores them and
answers nearest-neighbour queries by cosine similarity.

`build_index` creates a **fresh collection every time**, with a random suffix in the name.
Reusing a collection across configurations is the classic way to get results that look
plausible and are wrong, because the old vectors are still in there. The assertion on the
stored count is the guard: if Chroma holds a different number of vectors than the number
of chunks it was given, the run stops rather than producing quietly incorrect numbers.

One limitation to record now rather than discover later: MiniLM truncates its input at 256
word-pieces, roughly 1,000 characters. Chunks larger than that are embedded only in part
while being retrieved in full. Section 13 returns to this.

In [6]:
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import torch

EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.perf_counter()
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL,
                                   model_kwargs={"device": EMBED_DEVICE})
EMBED_LOAD_SECONDS = time.perf_counter() - t0

EMBED_DIM = len(embeddings.embed_query("dimension probe"))
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Dimensions      : {EMBED_DIM}")
print(f"Device          : {EMBED_DEVICE}")
print(f"Load time       : {EMBED_LOAD_SECONDS:.2f} s")


def build_index(chunk_size, chunk_overlap, top_k):
    '''Build a fresh vector index at the given configuration.

    Returns the retriever plus a timing and size breakdown. Every call creates a new
    collection, so no index is ever reused across configurations.
    '''
    split_chunks, t_split = split_documents(documents, chunk_size, chunk_overlap)

    name = f"{COLLECTION_NAME}_{uuid.uuid4().hex[:8]}"
    t0 = time.perf_counter()
    store = Chroma.from_documents(documents=split_chunks, embedding=embeddings,
                                  collection_name=name)
    t_index = time.perf_counter() - t0

    stored = store._collection.count()
    if stored != len(split_chunks):
        raise RuntimeError(
            f"Index holds {stored} vectors for {len(split_chunks)} chunks — stale index.")

    return {
        "retriever": store.as_retriever(search_kwargs={"k": top_k}),
        "store": store,
        "collection": name,
        "chunk_count": len(split_chunks),
        "mean_chunk_chars": sum(len(c.page_content) for c in split_chunks) / len(split_chunks),
        "split_seconds": t_split,
        "embed_index_seconds": t_index,
        "indexing_seconds": t_split + t_index,
        "stored_vectors": stored,
    }


built = build_index(CHUNK_SIZE, CHUNK_OVERLAP, TOP_K)
retriever = built["retriever"]
print(f"\nCollection      : {built['collection']}")
print(f"Chunks indexed  : {built['chunk_count']}  (stored vectors {built['stored_vectors']})")
print(f"Indexing time   : {built['indexing_seconds']:.2f} s")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model : sentence-transformers/all-MiniLM-L6-v2
Dimensions      : 384
Device          : cuda
Load time       : 29.59 s

Collection      : purpler_final_16d0a77c
Chunks indexed  : 125  (stored vectors 125)
Indexing time   : 0.68 s


## 7. Load the language model

`Qwen2.5-1.5B-Instruct` runs locally in this runtime. A hosted API model would have made
the generation stage unmeasurable from here and would have put the corpus on someone
else's server.

The dtype is chosen from the hardware rather than left to `"auto"`. `"auto"` reads the
model config and picks bfloat16, but bfloat16 has no native support on Turing cards such
as the T4 — it runs through emulation and is slower than float16. The check below uses
compute capability, which is the property that actually decides this.

Model loading is timed separately and reported as a one-time setup cost. It is not part
of answering a question and is never folded into a per-question measurement.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name(0)}  (compute capability {major}.{minor})")
    # Ampere (8.0) and later have native bfloat16; Turing (7.5) emulates it.
    DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    DTYPE = torch.float32

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, device_map=DEVICE_MAP, dtype=DTYPE, low_cpu_mem_usage=True)
model.eval()
MODEL_LOAD_SECONDS = time.perf_counter() - t0

LLM_DTYPE = str(next(model.parameters()).dtype)
LLM_DEVICE = str(model.get_input_embeddings().weight.device)
print(f"\nLoaded          : {LLM_MODEL}")
print(f"Selected dtype  : {LLM_DTYPE}")
print(f"Device          : {LLM_DEVICE}")
print(f"Load time       : {MODEL_LOAD_SECONDS:.2f} s  (one-time setup cost)")

CUDA available: True
GPU: Tesla T4  (compute capability 7.5)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Loaded          : Qwen/Qwen2.5-3B-Instruct
Selected dtype  : torch.float16
Device          : cuda:0
Load time       : 79.39 s  (one-time setup cost)


## 8. The RAG question-answering function

Retrieval, prompt assembly and generation are timed separately, because they behave
differently and a single blended number would hide that.

The prompt below is **version 3**. Two earlier versions were run end to end on the same
test set and the same configuration, and both are reported in Section 14. The second was
worse than the first, which is the more useful result of the two.

**Version 1** ended the user message with *"If the context does not answer the question,
reply: [refusal]."* Being the last thing read before generation, it dominated: four of the
nine answerable questions were refused, including one where all three retrieved chunks
contained evidence. Two of the three unanswerable questions were answered anyway, one of
them from the model's own knowledge of Singapore law.

**Version 2** removed that line and added rules against using outside knowledge and
against moving a figure from one rule to another. It made things worse — refusals rose
from four to five and mean answer quality fell. The diagnosis: six rules, nearly all of
them prohibitions, plus an explicit *"when you decline, reply exactly…"* gave a small
model a clean and safe way out. On A3 the model found the governing provision, quoted it
correctly, and then declined because the statute does not use the word "screenshot".

**Version 3 changes two things at once**, so the report attributes the result to the pair
rather than to either alone.

*A larger model.* Qwen2.5-**3B**-Instruct replaces the 1.5B. Instruction-following is
where small instruct models are weakest, and version 2 failed precisely at following
instructions — rule 2 forbade answering about foreign law and the model did it anyway. At
float16 the 3B weights are about 6 GB, which fits a T4 comfortably.

*A shorter prompt with worked examples instead of prohibitions.* Three rules rather than
six, and two short demonstrations — one answered, one declined. Small models imitate
examples far more reliably than they obey rule lists, and an example of a *good answer*
counterbalances the pull toward refusing that version 2 created.

In [8]:
REFUSAL = "The provided documents do not contain enough information to answer this question."

PROMPT_VERSION = 3

SYSTEM_PROMPT = (
    "You answer questions about Philippine consumer-remedy law from numbered context "
    "blocks.\n"
    "\n"
    "1. Answer from the blocks. If a block supports an answer even partly, give that "
    "answer - do not hold out for wording that matches the question exactly.\n"
    "2. If nothing in the blocks bears on the question, reply exactly: "
    f"\"{REFUSAL}\"\n"
    "3. Lead with the answer, then cite the blocks you used, like [2]. Two or three "
    "sentences."
)

EXAMPLE_ANSWERED = (
    "Context:\n[1] (example.txt)\nA service firm SHALL GUARANTEE workmanship for NOT "
    "LESS THAN NINETY (90) DAYS.\n\nQuestion: How long must a service firm guarantee "
    "its work?",
    "Not less than ninety (90) days [1]. The guarantee covers workmanship and must be "
    "stated in the invoice."
)

EXAMPLE_DECLINED = (
    "Context:\n[1] (example.txt)\nThe DTI shall develop an online dispute resolution "
    "platform WITHIN SIX (6) MONTHS of effectivity.\n\nQuestion: How long does that "
    "platform have to decide a complaint?",
    REFUSAL + " Block 1 sets a deadline for building the platform, not for deciding "
    "complaints through it."
)


def build_prompt(question, docs):
    blocks = []
    for i, d in enumerate(docs, start=1):
        src = os.path.basename(d.metadata.get("source", "?"))
        blocks.append(f"[{i}] ({src})\n{d.page_content}")
    context = "\n\n".join(blocks)
    # The refusal instruction lives in the system prompt only. In version 1 a trailing
    # refusal line was the last thing the model read and it refused four answerable
    # questions as a result.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": EXAMPLE_ANSWERED[0]},
        {"role": "assistant", "content": EXAMPLE_ANSWERED[1]},
        {"role": "user", "content": EXAMPLE_DECLINED[0]},
        {"role": "assistant", "content": EXAMPLE_DECLINED[1]},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)


def answer_question(question, active_retriever=None):
    r = active_retriever if active_retriever is not None else retriever

    t0 = time.perf_counter()
    docs = r.invoke(question)
    retrieval_seconds = time.perf_counter() - t0

    t0 = time.perf_counter()
    prompt = build_prompt(question, docs)
    prompt_seconds = time.perf_counter() - t0

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=DO_SAMPLE,
                             pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()          # CUDA is async; stop the clock after the work
    generation_seconds = time.perf_counter() - t0

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "question": question,
        "answer": answer,
        "documents": docs,
        "sources": [{"source": os.path.basename(d.metadata.get("source", "?")),
                     "chunk_start": d.metadata.get("start_index")} for d in docs],
        "retrieval_seconds": retrieval_seconds,
        "prompt_seconds": prompt_seconds,
        "generation_seconds": generation_seconds,
        "total_seconds": retrieval_seconds + prompt_seconds + generation_seconds,
        "output_tokens": int(len(new_tokens)),
    }


print("answer_question() ready")

answer_question() ready


## 9. Verify which device is actually in use

Selecting a GPU runtime does not place every operation on it. This cell reports where
things actually are rather than what was requested.

Chroma's similarity search runs on the host CPU either way. That is not a misconfiguration
— it is how Chroma works — but it means the retrieval timings in this notebook are CPU
timings even on a GPU runtime, and the report says so.

In [9]:
RUNTIME_LABEL = "GPU" if torch.cuda.is_available() else "CPU"

print("=" * 70)
print(f"RUNTIME: {RUNTIME_LABEL}")
print("=" * 70)
print(f"torch.cuda.is_available() : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name                  : {torch.cuda.get_device_name(0)}")
    print(f"GPU total memory          : "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
    print(f"GPU memory allocated      : {torch.cuda.memory_allocated() / 1024**2:.0f} MiB")
print(f"\nEmbedding model device    : {EMBED_DEVICE}")
print(f"Language model device     : {LLM_DEVICE}")
print(f"Language model dtype      : {LLM_DTYPE}")
print(f"Vector search (Chroma)    : CPU  - Chroma's similarity search is CPU-bound and is")
print(f"                                  not accelerated by a GPU runtime.")
print(f"\nModel load (setup cost)   : {MODEL_LOAD_SECONDS:.2f} s")

RUNTIME: GPU
torch.cuda.is_available() : True
GPU name                  : Tesla T4
GPU total memory          : 14.6 GiB
GPU memory allocated      : 5981 MiB

Embedding model device    : cuda
Language model device     : cuda:0
Language model dtype      : torch.float16
Vector search (Chroma)    : CPU  - Chroma's similarity search is CPU-bound and is
                                  not accelerated by a GPU runtime.

Model load (setup cost)   : 79.39 s


## 10. Test the chatbot end to end

One fixed question through the whole pipeline, to prove it works before anything is
measured. The question is hard-coded rather than typed, so *Run all* completes without
waiting for input and the answer is preserved in the submitted notebook.

In [10]:
test_question = ("Under the Philippine Lemon Law, how many separate repair attempts for "
                 "the same complaint must fail before a consumer may invoke their rights?")

print("QUESTION:", test_question, "\n")
result = answer_question(test_question)

print("ANSWER")
print("-" * 70)
print(result["answer"])

print("\nSOURCES")
print("-" * 70)
for i, s in enumerate(result["sources"], start=1):
    print(f"  [{i}] {s['source']}  (chunk start {s['chunk_start']})")

print("\nTIMING")
print("-" * 70)
print(f"  retrieval  : {result['retrieval_seconds']*1000:7.1f} ms")
print(f"  prompt     : {result['prompt_seconds']*1000:7.1f} ms")
print(f"  generation : {result['generation_seconds']:7.2f} s   "
      f"({result['output_tokens']} tokens)")
print(f"  total      : {result['total_seconds']:7.2f} s")

QUESTION: Under the Philippine Lemon Law, how many separate repair attempts for the same complaint must fail before a consumer may invoke their rights? 

ANSWER
----------------------------------------------------------------------
Under the Philippine Lemon Law, a consumer must have at least four (4) separate repair attempts for the same complaint fail before invoking their rights [1].

SOURCES
----------------------------------------------------------------------
  [1] 03_RA10642_Lemon_Law.txt  (chunk start 5701)
  [2] 03_RA10642_Lemon_Law.txt  (chunk start 6243)
  [3] 03_RA10642_Lemon_Law.txt  (chunk start 15279)

TIMING
----------------------------------------------------------------------
  retrieval  :    64.9 ms
  prompt     :    11.5 ms
  generation :    1.98 s   (32 tokens)
  total      :    2.06 s


## 11. The test set and the scoring functions

Twelve questions, authored by R. K. J. Leliza: nine answerable across three difficulty
levels, three unanswerable. The CSV is read directly, so the questions used here and the
questions documented in the data package cannot drift apart.

### The three scoring decisions

**`keep_default_na=False`.** Pandas turns an empty cell into `NaN`, a float. Calling
`.split(";")` on it raises `AttributeError` before any guard can run, and the obvious
workaround — `str(keys)` — is worse: it produces the string `"nan"`, which is a perfectly
good key that nothing will ever match, so the unanswerable rows silently score 0.0
instead of being undefined. Reading blanks as empty strings avoids both.

**`;` separates required facts, `|` separates acceptable variants.** `A1`'s key is
`thirty|30;replacement|replace;reimbursement|refund;price reduction` — four separate
facts, each satisfied by any one of its variants. An answer naming only a refund scores
0.25, which is the right verdict on an answer that names one of three remedies the
statute offers.

**Digit-only variants match on word boundaries.** The bare variant `4` was matching inside
"7394", so an answer that only name-dropped the statutes scored 1.00 on E1 without ever
saying "four". Purely numeric variants are now matched whole; everything else stays a
plain substring match, which keeps stems such as `exclud` working against both "excludes"
and "excluded".

### Refusal is measured separately, in both directions

Hallucination and over-refusal are opposite failures and both matter. `Declined_Correctly`
applies to the three unanswerable questions; `Wrongly_Declined` applies to the nine
answerable ones. Neither is mixed into the answer-quality mean.

One subtlety, found in a dry run before this notebook was ever sent to Colab: detecting a
refusal by phrase alone is too crude. Models routinely answer correctly and then add a
hedge — *"...the documents do not specify the filing fee"* — and a plain phrase search
marks that a refusal. So for an answerable question a refusal is counted only when the
phrase is present **and** the answer satisfies none of its keys. An answer that earned
points did not refuse, whatever else it said.

In [11]:
import pandas as pd, numpy as np, re

testset = pd.read_csv(TESTSET_CSV, encoding="utf-8-sig", keep_default_na=False)
QUESTIONS = testset.to_dict("records")

ANSWERABLE = [q for q in QUESTIONS if q["Type"].strip().lower() == "answerable"]
UNANSWERABLE = [q for q in QUESTIONS if q["Type"].strip().lower() == "unanswerable"]

print(f"Test set: {len(QUESTIONS)} questions "
      f"({len(ANSWERABLE)} answerable, {len(UNANSWERABLE)} unanswerable)")
for q in QUESTIONS:
    n_anchors = sum(1 for c in ("Evidence_Anchor_1", "Evidence_Anchor_2", "Evidence_Anchor_3")
                    if str(q.get(c, "")).strip())
    print(f"  {q['ID']:<4}{q['Difficulty']:<14}{n_anchors} anchors   {q['Question'][:62]}")


def _norm(text):
    return re.sub(r"\s+", " ", str(text)).strip().upper()


def anchors_of(q):
    return [str(q.get(c, "")).strip()
            for c in ("Evidence_Anchor_1", "Evidence_Anchor_2", "Evidence_Anchor_3")
            if str(q.get(c, "")).strip()]


def chunk_is_relevant(chunk_text, q):
    '''A chunk is relevant when it contains at least one verbatim anchor.'''
    flat = _norm(chunk_text)
    return any(_norm(a) in flat for a in anchors_of(q))


def retrieval_scores(docs, q):
    '''Recall@k and Precision@k. Undefined (None) when the question has no anchors.'''
    anchors = anchors_of(q)
    if not anchors:
        return None, None
    flat = [_norm(d.page_content) for d in docs]
    found = sum(1 for a in anchors if any(_norm(a) in f for f in flat))
    relevant_chunks = sum(1 for f in flat if any(_norm(a) in f for a in anchors))
    return found / len(anchors), relevant_chunks / len(docs)


def answer_quality(answer, answer_keys):
    '''Fraction of required facts present. Undefined (None) when there are no keys.'''
    keys = [k.strip() for k in str(answer_keys).split(";") if k.strip()]
    if not keys:
        return None
    low = str(answer).lower()

    def hit(variant):
        v = variant.strip().lower()
        if not v:
            return False
        if v.isdigit():                      # bare digits must match whole
            return re.search(rf"\b{re.escape(v)}\b", low) is not None
        return v in low                      # substrings keep stems like 'exclud' working

    return sum(any(hit(v) for v in k.split("|")) for k in keys) / len(keys)


DECLINE_PHRASES = (
    "do not contain", "does not contain", "not contain enough", "not provide",
    "does not provide", "no mention", "not mention", "does not specify", "not specify",
    "no information", "not available", "does not say", "does not state", "not stated",
    "cannot find", "insufficient", "not enough information", "outside the scope",
)


def declined(answer):
    low = str(answer).lower()
    return any(p in low for p in DECLINE_PHRASES)


print("\nScoring functions ready: retrieval_scores(), answer_quality(), declined()")

Test set: 12 questions (9 answerable, 3 unanswerable)
  E1  Easy          1 anchors   Under the Philippine Lemon Law, how many separate repair attem
  E2  Easy          1 anchors   How long must a service firm guarantee its workmanship and rep
  E3  Easy          2 anchors   Where do I send a consumer complaint against an online seller?
  A1  Average       3 anchors   The store has had my defective appliance for over a month and 
  A2  Average       2 anchors   My Shopee order arrived broken. Can I file with the DTI right 
  A3  Average       3 anchors   Are screenshots of my chat with the seller and my emailed rece
  D1  Difficult     3 anchors   My brand new motorcycle has been in for the same repair four t
  D2  Difficult     3 anchors   I bought a defective second-hand laptop from a private individ
  D3  Difficult     3 anchors   The listing said the item was brand new but it arrived clearly
  U1  Unanswerable  0 anchors   What is the warranty period for home appliances under Singa

## 12. Chunking experiment

Five configurations, every one of the twelve questions retrieved against each. The project
requires at least three; five lets each parameter be isolated rather than confounded.

| Run | Size | Overlap | k | What it isolates |
|---|---|---|---|---|
| A | 400 | 50 | 3 | small chunks, 12.5% overlap |
| B | 800 | 100 | 3 | the baseline |
| C | 1200 | 150 | 3 | large chunks, same overlap ratio |
| D | 800 | 200 | 3 | **overlap** — same size as B, double the overlap |
| E | 800 | 100 | 5 | **retrieval depth** — same index as B, k raised to 5 |

A, B and C hold the overlap ratio constant and vary size, so the size effect is clean. D
changes only the overlap. E changes only `k`. Without that structure a difference between
two runs could be attributed to either parameter.

Generation runs on a three-question subset — one easy, one difficult, one unanswerable —
because generating for all twelve at five configurations would cost an hour for a result
the full evaluation in Section 14 produces properly. Retrieval, which is cheap, runs on
all twelve.

In [12]:
RUNS = [
    ("A",  400,  50, 3),
    ("B",  800, 100, 3),
    ("C", 1200, 150, 3),
    ("D",  800, 200, 3),
    ("E",  800, 100, 5),
]
GEN_SUBSET = ["E1", "D1", "U1"]

print("Warm-up (discarded — the first generation pays one-off CUDA setup)")
_ = answer_question("What is a brand new motor vehicle?")
print("warm-up done\n")

sweep_rows, sweep_detail = [], []

for run, size, overlap, k in RUNS:
    b = build_index(size, overlap, k)
    r = b["retriever"]

    recalls, precisions, qualities = [], [], []
    retrieval_times, generation_times = [], []

    for q in QUESTIONS:
        t0 = time.perf_counter()
        docs = r.invoke(q["Question"])
        retrieval_times.append(time.perf_counter() - t0)

        rec, prec = retrieval_scores(docs, q)
        if rec is not None:
            recalls.append(rec)
            precisions.append(prec)

        aq = None
        if q["ID"] in GEN_SUBSET:
            res = answer_question(q["Question"], active_retriever=r)
            generation_times.append(res["generation_seconds"])
            aq = answer_quality(res["answer"], q["Answer_Keys"])
            if aq is not None:
                qualities.append(aq)

        sweep_detail.append({
            "Run": run, "Chunk_Size": size, "Overlap": overlap, "Top_K": k,
            "Question_ID": q["ID"], "Type": q["Type"], "Difficulty": q["Difficulty"],
            "Recall_at_k": rec, "Precision_at_k": prec, "Answer_Quality": aq,
        })

    sweep_rows.append({
        "Run": run, "Chunk_Size": size, "Overlap": overlap, "Top_K": k,
        "Total_Chunks": b["chunk_count"],
        "Mean_Chunk_Chars": round(b["mean_chunk_chars"], 1),
        "Split_s": round(b["split_seconds"], 3),
        "Embed_Index_s": round(b["embed_index_seconds"], 3),
        "Indexing_s": round(b["indexing_seconds"], 3),
        "Mean_Retrieval_ms": round(np.mean(retrieval_times) * 1000, 1),
        "Mean_Generation_s": round(np.mean(generation_times), 2) if generation_times else None,
        "Recall_at_k": round(np.mean(recalls), 4),
        "Precision_at_k": round(np.mean(precisions), 4),
        "Answer_Quality": round(np.mean(qualities), 4) if qualities else None,
    })

    row = sweep_rows[-1]
    print(f"  {run}  {size:>5}/{overlap:<4} k={k}  "
          f"chunks {row['Total_Chunks']:>4}  index {row['Indexing_s']:>6.2f}s  "
          f"retr {row['Mean_Retrieval_ms']:>5.1f}ms  "
          f"R@k {row['Recall_at_k']:.3f}  P@k {row['Precision_at_k']:.3f}  "
          f"ansQ {row['Answer_Quality']}")

sweep = pd.DataFrame(sweep_rows)
sweep_detail_df = pd.DataFrame(sweep_detail)

print("\n" + "=" * 78)
print("CHUNKING EXPERIMENT")
print("=" * 78)
print(sweep.to_string(index=False))

Warm-up (discarded — the first generation pays one-off CUDA setup)
warm-up done

  A    400/50   k=3  chunks  268  index   0.55s  retr  11.1ms  R@k 0.333  P@k 0.111  ansQ 0.6667
  B    800/100  k=3  chunks  125  index   0.49s  retr  12.4ms  R@k 0.333  P@k 0.111  ansQ 0.5
  C   1200/150  k=3  chunks   78  index   0.39s  retr  19.9ms  R@k 0.593  P@k 0.296  ansQ 0.5
  D    800/200  k=3  chunks  131  index   0.55s  retr  19.0ms  R@k 0.296  P@k 0.148  ansQ 0.6667
  E    800/100  k=5  chunks  125  index   0.60s  retr  10.3ms  R@k 0.407  P@k 0.111  ansQ 0.5

CHUNKING EXPERIMENT
Run  Chunk_Size  Overlap  Top_K  Total_Chunks  Mean_Chunk_Chars  Split_s  Embed_Index_s  Indexing_s  Mean_Retrieval_ms  Mean_Generation_s  Recall_at_k  Precision_at_k  Answer_Quality
  A         400       50      3           268             274.2    0.003          0.547       0.551               11.1               2.97       0.3333          0.1111          0.6667
  B         800      100      3           125           

## 13. Selecting the configuration

The rule, fixed before the numbers were seen: rank on retrieval quality, keep everything
within a small tolerance of the best, and settle the remainder on indexing cost. Retrieval
quality is the mean of Recall@k and Precision@k, because a configuration that retrieves
the evidence inside a pile of noise has not really succeeded.

The justification is written out in full, including the parameters that made things
worse. A sweep that reports only the winner hides most of what it learned.

In [13]:
Q_TOL = 0.02

sel = sweep.copy()
sel["Retrieval_Quality"] = sel[["Recall_at_k", "Precision_at_k"]].mean(axis=1).round(4)
sel = sel.sort_values("Retrieval_Quality", ascending=False)

print("=" * 78)
print("SELECTION")
print("=" * 78)
print(sel[["Run", "Chunk_Size", "Overlap", "Top_K", "Total_Chunks",
           "Recall_at_k", "Precision_at_k", "Retrieval_Quality",
           "Indexing_s", "Mean_Retrieval_ms"]].to_string(index=False))

best = sel["Retrieval_Quality"].max()
near = sel[sel["Retrieval_Quality"] >= best - Q_TOL]
winner = near.sort_values("Indexing_s").iloc[0]

print(f"\nBest retrieval quality {best:.4f}; within {Q_TOL} of it: "
      f"{', '.join(near['Run'].tolist())}")
print(f"Of those, cheapest to index: Run {winner['Run']} at {winner['Indexing_s']:.2f} s")

REC_SIZE = int(winner["Chunk_Size"])
REC_OVERLAP = int(winner["Overlap"])
REC_K = int(winner["Top_K"])

print("\n" + "=" * 78)
print("SELECTED CONFIGURATION")
print("=" * 78)
print(f"  Run {winner['Run']}  -  chunk size {REC_SIZE}, overlap {REC_OVERLAP}, k = {REC_K}")
print(f"  Chunks {int(winner['Total_Chunks'])} | indexing {winner['Indexing_s']:.2f} s")
print(f"  Recall@k {winner['Recall_at_k']:.3f} | Precision@k {winner['Precision_at_k']:.3f}")

# --- what the other parameters did, stated whether or not it flatters the result ---
b_row = sweep[sweep.Run == "B"].iloc[0]
d_row = sweep[sweep.Run == "D"].iloc[0]
e_row = sweep[sweep.Run == "E"].iloc[0]

print("\nJUSTIFICATION")
print("-" * 78)
print(f"  Chunk size (A->B->C): R@k {sweep[sweep.Run=='A'].iloc[0]['Recall_at_k']:.3f} -> "
      f"{b_row['Recall_at_k']:.3f} -> {sweep[sweep.Run=='C'].iloc[0]['Recall_at_k']:.3f}")
print(f"  Overlap  (B->D, 100->200): R@k {b_row['Recall_at_k']:.3f} -> {d_row['Recall_at_k']:.3f}, "
      f"P@k {b_row['Precision_at_k']:.3f} -> {d_row['Precision_at_k']:.3f}")
print(f"  Top-k    (B->E, 3->5):     R@k {b_row['Recall_at_k']:.3f} -> {e_row['Recall_at_k']:.3f}, "
      f"P@k {b_row['Precision_at_k']:.3f} -> {e_row['Precision_at_k']:.3f}")
print("\n  Note: chunks larger than ~1,000 characters exceed the 256 word-piece input")
print("  limit of all-MiniLM-L6-v2, so they are embedded in part and retrieved in full.")
print("  Any advantage at 1,200 characters is therefore partly payload, not representation.")

final = build_index(REC_SIZE, REC_OVERLAP, REC_K)
retriever = final["retriever"]
print(f"\nRebuilt the index at the selected configuration: "
      f"{final['chunk_count']} chunks in {final['indexing_seconds']:.2f} s")

SELECTION
Run  Chunk_Size  Overlap  Top_K  Total_Chunks  Recall_at_k  Precision_at_k  Retrieval_Quality  Indexing_s  Mean_Retrieval_ms
  C        1200      150      3            78       0.5926          0.2963             0.4444       0.392               19.9
  E         800      100      5           125       0.4074          0.1111             0.2592       0.599               10.3
  A         400       50      3           268       0.3333          0.1111             0.2222       0.551               11.1
  B         800      100      3           125       0.3333          0.1111             0.2222       0.491               12.4
  D         800      200      3           131       0.2963          0.1481             0.2222       0.547               19.0

Best retrieval quality 0.4444; within 0.02 of it: C
Of those, cheapest to index: Run C at 0.39 s

SELECTED CONFIGURATION
  Run C  -  chunk size 1200, overlap 150, k = 3
  Chunks 78 | indexing 0.39 s
  Recall@k 0.593 | Precision@k 0.296

JU

## 14. Full evaluation of all twelve questions

**Deliverable 5.** Every question answered at the selected configuration, with the
reference answer alongside the generated one and a comparative assessment of each.

The blank cells are deliberate. Recall, precision and answer quality are undefined for the
three unanswerable questions, and writing zero there would understate the mean by a
quarter. Refusal behaviour is reported instead, in its own columns, in both directions.

In [14]:
eval_rows = []
print(f"Evaluating {len(QUESTIONS)} questions at {REC_SIZE}/{REC_OVERLAP}, k={REC_K}\n")

for q in QUESTIONS:
    res = answer_question(q["Question"])
    rec, prec = retrieval_scores(res["documents"], q)
    aq = answer_quality(res["answer"], q["Answer_Keys"])
    phrase_present = declined(res["answer"])
    unanswerable = q["Type"].strip().lower() == "unanswerable"
    # A hedge attached to a real answer is not a refusal: require that nothing scored.
    did_decline = phrase_present if unanswerable else (phrase_present and not aq)

    eval_rows.append({
        "ID": q["ID"],
        "Type": q["Type"],
        "Difficulty": q["Difficulty"],
        "Question": q["Question"],
        "Reference_Answer": q["Reference_Answer"],
        "Chatbot_Answer": res["answer"],
        "Sources_Retrieved": "; ".join(s["source"] for s in res["sources"]),
        "Recall_at_k": rec,
        "Precision_at_k": prec,
        "Answer_Quality": aq,
        "Declined_Correctly": did_decline if unanswerable else None,
        "Wrongly_Declined": did_decline if not unanswerable else None,
        "Assessment": (
            ("correct refusal" if did_decline else "FAILED TO DECLINE") if unanswerable
            else ("wrongly refused" if did_decline
                  else "full" if aq == 1.0 else "partial" if aq and aq > 0 else "missed")),
        "Retrieval_ms": round(res["retrieval_seconds"] * 1000, 1),
        "Generation_s": round(res["generation_seconds"], 2),
        "Output_Tokens": res["output_tokens"],
    })

    r = eval_rows[-1]
    fmt = lambda v: "    -" if v is None else f"{v:5.2f}"
    print(f"  {q['ID']:<4}{q['Difficulty']:<14}R@k {fmt(rec)}  P@k {fmt(prec)}  "
          f"ansQ {fmt(aq)}  {r['Assessment']}")

evaluation = pd.DataFrame(eval_rows)

scored = evaluation[evaluation.Answer_Quality.notna()]
decliners = evaluation[evaluation.Declined_Correctly.notna()]

print("\n" + "=" * 78)
print("EVALUATION SUMMARY")
print("=" * 78)
print(f"  Scored questions          : {len(scored)}  (answerable only)")
print(f"  Mean Recall@k             : {scored.Recall_at_k.mean():.4f}")
print(f"  Mean Precision@k          : {scored.Precision_at_k.mean():.4f}")
print(f"  Mean Answer Quality       : {scored.Answer_Quality.mean():.4f}")
print(f"  Fully correct             : {(scored.Answer_Quality == 1.0).sum()} of {len(scored)}")
print(f"  Wrongly declined          : {int(scored.Wrongly_Declined.sum())} of {len(scored)}")
print(f"  Unanswerable questions    : {len(decliners)}")
print(f"  Declined correctly        : {int(decliners.Declined_Correctly.sum())} of {len(decliners)}")
print(f"  Mean generation time      : {evaluation.Generation_s.mean():.2f} s")

evaluation.to_csv(f"{GROUP_NAME}_Chatbot_Evaluation.csv", index=False)
print(f"\nWritten: {GROUP_NAME}_Chatbot_Evaluation.csv")

# ---- prompt iteration: version 1, measured on this same test set and configuration ----
# v1 and v2 were measured on this same test set, configuration and metric code.
PRIOR = {
    "v1 (1.5B, 5 rules)":  {"Mean Answer Quality": 0.4074, "Fully correct (of 9)": 3,
                            "Wrongly declined (of 9)": 4, "Declined correctly (of 3)": 1},
    "v2 (1.5B, 6 rules)":  {"Mean Answer Quality": 0.3704, "Fully correct (of 9)": 3,
                            "Wrongly declined (of 9)": 5, "Declined correctly (of 3)": 1},
}
THIS_RUN = {
    "Mean Answer Quality": round(scored.Answer_Quality.mean(), 4),
    "Fully correct (of 9)": int((scored.Answer_Quality == 1.0).sum()),
    "Wrongly declined (of 9)": int(scored.Wrongly_Declined.sum()),
    "Declined correctly (of 3)": int(decliners.Declined_Correctly.sum()),
}
this_label = f"v{PROMPT_VERSION} ({LLM_MODEL.split('/')[-1].replace('-Instruct','')}, 3 rules + 2 examples)"

iteration = pd.DataFrame([
    dict(Metric=m, **{k: v[m] for k, v in PRIOR.items()}, **{this_label: THIS_RUN[m]})
    for m in THIS_RUN
])

print("\n" + "=" * 78)
print("PROMPT ITERATION  (same test set, same chunking configuration, same metric code)")
print("=" * 78)
print(iteration.to_string(index=False))
print("\n  v1: refusal instruction at the end of the user message.")
print("  v2: that line removed, two prohibitions added - WORSE, refusals rose 4 -> 5.")
print("  v3: larger model, three rules instead of six, two worked examples")
print("      instead of a list of prohibitions.")

iteration.to_csv(f"{GROUP_NAME}_Prompt_Iteration.csv", index=False)
print(f"\nWritten: {GROUP_NAME}_Prompt_Iteration.csv")

Evaluating 12 questions at 1200/150, k=3

  E1  Easy          R@k  1.00  P@k  0.33  ansQ  1.00  full
  E2  Easy          R@k  1.00  P@k  0.33  ansQ  1.00  full
  E3  Easy          R@k  1.00  P@k  0.33  ansQ  1.00  full
  A1  Average       R@k  1.00  P@k  0.33  ansQ  0.50  partial
  A2  Average       R@k  0.00  P@k  0.00  ansQ  0.33  partial
  A3  Average       R@k  1.00  P@k  1.00  ansQ  0.33  partial
  D1  Difficult     R@k  0.00  P@k  0.00  ansQ  0.00  missed
  D2  Difficult     R@k  0.33  P@k  0.33  ansQ  0.00  missed
  D3  Difficult     R@k  0.00  P@k  0.00  ansQ  0.67  partial
  U1  Unanswerable  R@k     -  P@k     -  ansQ     -  correct refusal
  U2  Unanswerable  R@k     -  P@k     -  ansQ     -  correct refusal
  U3  Unanswerable  R@k     -  P@k     -  ansQ     -  FAILED TO DECLINE

EVALUATION SUMMARY
  Scored questions          : 9  (answerable only)
  Mean Recall@k             : 0.5926
  Mean Precision@k          : 0.2963
  Mean Answer Quality       : 0.5370
  Fully correct  

## 15. Experiment logs and checkpoints

**Deliverable 4. Owned by S. J. C. Dizon.**

Eight sheets, one per pipeline stage, written from the variables this run actually
produced rather than typed up afterwards. The instructions ask that the claims in the
paper be *auditable*; this workbook is what makes them so.

Check before sending it on:

- Every sheet has rows. An empty sheet means that stage did not run.
- `04_Vector_Database` shows stored vectors **equal to** the chunk count.
- `07_Chatbot_Evaluation` has all twelve questions, with blanks — not zeros — in the
  score columns for U1–U3.
- The timestamp in `08_Environment` matches when the run actually happened.

In [15]:
import platform, datetime, transformers, chromadb
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

LOGS_XLSX = f"{GROUP_NAME}_Experiment_Logs.xlsx"
RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

doc_sheet = pd.DataFrame(DOC_STATS)
doc_sheet.loc[len(doc_sheet)] = {"Document": "TOTAL", "Characters": SOURCE_CHARS}

chunk_sheet = sweep[["Run", "Chunk_Size", "Overlap", "Top_K", "Total_Chunks",
                     "Mean_Chunk_Chars", "Split_s", "Embed_Index_s", "Indexing_s"]].copy()

embed_sheet = pd.DataFrame([{
    "Model": EMBEDDING_MODEL, "Dimensions": EMBED_DIM, "Device": EMBED_DEVICE,
    "Model_Load_s": round(EMBED_LOAD_SECONDS, 2),
    "Corpus_Characters": SOURCE_CHARS,
    "Embed_Index_s_at_selected": round(final["embed_index_seconds"], 3),
    "Chunks_Embedded": final["chunk_count"],
    "Note": "MiniLM truncates at 256 word-pieces (~1,000 chars)",
}])

vectordb_sheet = pd.DataFrame([{
    "Backend": "Chroma (in-memory)",
    "Chromadb_Version": getattr(chromadb, "__version__", "unknown"),
    "Collection": final["collection"],
    "Chunks_Supplied": final["chunk_count"],
    "Vectors_Stored": final["stored_vectors"],
    "Counts_Match": final["stored_vectors"] == final["chunk_count"],
    "Build_s": round(final["embed_index_seconds"], 3),
    "Search_Device": "CPU (Chroma similarity search is CPU-bound)",
}])

retrieval_sheet = evaluation[["ID", "Type", "Difficulty", "Question",
                              "Sources_Retrieved", "Recall_at_k", "Precision_at_k",
                              "Retrieval_ms"]].copy()

llm_sheet = pd.DataFrame([{
    "Model": LLM_MODEL, "Dtype": LLM_DTYPE, "Device": LLM_DEVICE,
    "Transformers_Version": transformers.__version__,
    "Model_Load_s": round(MODEL_LOAD_SECONDS, 2),
    "Max_New_Tokens": MAX_NEW_TOKENS, "Do_Sample": DO_SAMPLE,
    "Mean_Generation_s": round(evaluation.Generation_s.mean(), 3),
    "Total_Output_Tokens": int(evaluation.Output_Tokens.sum()),
}])

env_sheet = pd.DataFrame([
    {"Key": "Run timestamp", "Value": RUN_TIMESTAMP},
    {"Key": "Runtime", "Value": RUNTIME_LABEL},
    {"Key": "GPU", "Value": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"},
    {"Key": "Python", "Value": platform.python_version()},
    {"Key": "Platform", "Value": platform.platform()},
    {"Key": "torch", "Value": torch.__version__},
    {"Key": "CUDA (torch)", "Value": str(torch.version.cuda)},
    {"Key": "transformers", "Value": transformers.__version__},
    {"Key": "chromadb", "Value": getattr(chromadb, "__version__", "unknown")},
    {"Key": "Embedding model", "Value": EMBEDDING_MODEL},
    {"Key": "Language model", "Value": LLM_MODEL},
    {"Key": "Selected configuration", "Value": f"{REC_SIZE}/{REC_OVERLAP}, k={REC_K}"},
    {"Key": "Corpus characters", "Value": SOURCE_CHARS},
    {"Key": "Test questions", "Value": f"{len(QUESTIONS)} ({len(ANSWERABLE)} answerable, "
                                       f"{len(UNANSWERABLE)} unanswerable)"},
    {"Key": "Pre-flight check", "Value": "passed" if PREFLIGHT_PASSED else str(PREFLIGHT_PASSED)},
    {"Key": "Prompt version", "Value": PROMPT_VERSION},
    {"Key": "Logs owner", "Value": "S. J. C. Dizon"},
])

SHEETS = {
    "01_Document_Processing": doc_sheet,
    "02_Chunking": chunk_sheet,
    "03_Embedding_Generation": embed_sheet,
    "04_Vector_Database": vectordb_sheet,
    "05_Retrieval": retrieval_sheet,
    "06_LLM_Integration": llm_sheet,
    "07_Chatbot_Evaluation": evaluation,
    "08_Environment": env_sheet,
    "09_Prompt_Iteration": iteration,
}

HEAD_FILL = PatternFill("solid", fgColor="4A3A6B")
HEAD_FONT = Font(bold=True, color="FFFFFF")

with pd.ExcelWriter(LOGS_XLSX, engine="openpyxl") as writer:
    for name, df in SHEETS.items():
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for cell in ws[1]:
            cell.fill, cell.font = HEAD_FILL, HEAD_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center")
        ws.freeze_panes = "A2"
        for i, col in enumerate(df.columns, start=1):
            longest = max([len(str(col))] + [len(str(v)) for v in df[col].head(60)])
            ws.column_dimensions[get_column_letter(i)].width = min(max(longest + 2, 10), 60)

print(f"Written: {LOGS_XLSX}")
for name, df in SHEETS.items():
    print(f"  {name:<26}{len(df):>4} rows")

print("\nACCEPTANCE CHECKS")
print("-" * 70)
ok = True
for name, df in SHEETS.items():
    if len(df) == 0:
        print(f"  FAIL  {name} is empty"); ok = False
if not bool(vectordb_sheet.Counts_Match.iloc[0]):
    print("  FAIL  stored vectors != chunk count"); ok = False
else:
    print(f"  ok    stored vectors == chunk count ({final['stored_vectors']})")
if len(evaluation) != 12:
    print(f"  FAIL  evaluation has {len(evaluation)} rows, expected 12"); ok = False
else:
    print("  ok    evaluation has 12 questions")
blank_ok = evaluation[evaluation.Type.str.lower() == "unanswerable"][
    ["Recall_at_k", "Precision_at_k", "Answer_Quality"]].isna().all().all()
print(f"  {'ok  ' if blank_ok else 'FAIL'}  U1-U3 score columns are blank, not zero")
ok = ok and blank_ok
print("-" * 70)
print("ALL ACCEPTANCE CHECKS PASSED" if ok else "SOME CHECKS FAILED - do not submit yet")

Written: PurpleR_Experiment_Logs.xlsx
  01_Document_Processing       6 rows
  02_Chunking                  5 rows
  03_Embedding_Generation      1 rows
  04_Vector_Database           1 rows
  05_Retrieval                12 rows
  06_LLM_Integration           1 rows
  07_Chatbot_Evaluation       12 rows
  08_Environment              17 rows
  09_Prompt_Iteration          4 rows

ACCEPTANCE CHECKS
----------------------------------------------------------------------
  ok    stored vectors == chunk count (78)
  ok    evaluation has 12 questions
  ok    U1-U3 score columns are blank, not zero
----------------------------------------------------------------------
ALL ACCEPTANCE CHECKS PASSED


## 16. The working chatbot

The experiment is finished; this is the deliverable the experiment was for. The index is
the one Section 13 selected, and `ask()` answers against it.

The questions are fixed so that *Run all* completes without waiting for input and the
answers survive in the submitted notebook. The last one is out of scope on purpose.

In [16]:
def ask(question):
    '''Answer one question at the selected configuration, with sources and timing.'''
    res = answer_question(question)
    print(f"YOU : {question}\n")
    print("ASSISTANT:")
    print(res["answer"])
    print("\nSOURCES:")
    for i, s in enumerate(res["sources"], start=1):
        print(f"  [{i}] {s['source']}  (chunk start {s['chunk_start']})")
    print(f"\nretrieval {res['retrieval_seconds']*1000:.1f} ms | "
          f"generation {res['generation_seconds']:.2f} s | "
          f"{res['output_tokens']} tokens | {RUNTIME_LABEL}")
    return res


for _q in [
    "My brand new car has been in for the same repair four times. What are my options?",
    "What must I do before filing a complaint about an online purchase?",
    "Who won the 2019 Philippine basketball championship?",
]:
    ask(_q)
    print("\n" + "=" * 78 + "\n")

YOU : My brand new car has been in for the same repair four times. What are my options?

ASSISTANT:
If your brand new car has been in for the same repair four times, you can invoke your rights under the Lemon Law. You must also provide notice to the manufacturer, distributor, authorized dealer, or retailer within thirty (30) calendar days of receiving the vehicle after the last repair attempt. If the vehicle is not returned for further repair within this period, the repair is deemed successful, and you may be eligible for additional remedies under Sections 5 and 6 of the act.

SOURCES:
  [1] 03_RA10642_Lemon_Law.txt  (chunk start 14797)
  [2] 03_RA10642_Lemon_Law.txt  (chunk start 7783)
  [3] 03_RA10642_Lemon_Law.txt  (chunk start 5412)

retrieval 12.8 ms | generation 10.49 s | 98 tokens | GPU


YOU : What must I do before filing a complaint about an online purchase?

ASSISTANT:
Before filing a complaint about an online purchase, you must first avail of the internal redress mechanism o

## 17. Download the deliverables

Four files. The workbook and the evaluation CSV are the submitted artefacts; the sweep
exports are the raw numbers behind the chunking section of the report.

In [17]:
from google.colab import files as colab_files

sweep.to_csv(f"{GROUP_NAME}_Chunking_Experiment.csv", index=False)
sweep_detail_df.to_csv(f"{GROUP_NAME}_Chunking_Detail.csv", index=False)

with pd.ExcelWriter(f"{GROUP_NAME}_Chunking_Experiment.xlsx", engine="openpyxl") as w:
    sweep.to_excel(w, sheet_name="Summary", index=False)
    sweep_detail_df.to_excel(w, sheet_name="Per_Question", index=False)
    sel[["Run", "Chunk_Size", "Overlap", "Top_K", "Recall_at_k", "Precision_at_k",
         "Retrieval_Quality", "Indexing_s"]].to_excel(w, sheet_name="Selection", index=False)

OUTPUTS = [
    LOGS_XLSX,
    f"{GROUP_NAME}_Chatbot_Evaluation.csv",
    f"{GROUP_NAME}_Prompt_Iteration.csv",
    f"{GROUP_NAME}_Chunking_Experiment.xlsx",
    f"{GROUP_NAME}_Chunking_Experiment.csv",
    f"{GROUP_NAME}_Chunking_Detail.csv",
]

for f in OUTPUTS:
    if os.path.exists(f):
        colab_files.download(f)
        print("downloaded:", f)
    else:
        print("MISSING   :", f)

print(f"\nRun complete on {RUNTIME_LABEL} at {RUN_TIMESTAMP}.")
print("Send these to Mark together with the screenshots and the run notes.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: PurpleR_Experiment_Logs.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: PurpleR_Chatbot_Evaluation.csv


... (8 outputs left out)

downloaded: PurpleR_Chunking_Experiment.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: PurpleR_Chunking_Detail.csv

Run complete on GPU at 2026-11-07 12:18:53.
Send these to Mark together with the screenshots and the run notes.
